## Generate Heston Volatility Grids

In [ ]:
import QuantLib as ql
from pydoe import lhs
import math
import pandas as pd
import random as rd
from py_vollib.black_scholes.implied_volatility import implied_volatility
from tqdm.notebook import tqdm, trange
import numpy as np
import os
import multiprocessing as mp

In [ ]:
# Function to generate LHS samples
def generate_lhs_samples(samples, dimensions):
    return lhs(dimensions, samples)

# Function to scale LHS samples according to the specified bounds
def scale_lhs_samples(lhs_samples, bounds):
    return np.array([sample * (bound[1] - bound[0]) + bound[0] for sample, bound in zip(lhs_samples.T, bounds)]).T

In [ ]:
def generateData(ID, pkgsize, filename):
    np.random.seed(ID)
    bounds = [(0.0, 5.0), (-1.0, 0.0), (0.0, 10.0), (0.0, 1.0), (0.0, 1.0)]
    S0 = 1.0
    q = 0.0
    r = 0.0
    strikes = [0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.2,1.3,1.4,1.5]
    maturities = [0.1,0.3,0.6,0.9,1.2,1.5,1.8,2.0]
    option_data = []
    lhs_samples = generate_lhs_samples(pkgsize, len(bounds))
    scaled_samples = scale_lhs_samples(lhs_samples, bounds)

    for i in range(pkgsize):
        eta, rho, lamda, vbar, V0 = scaled_samples[i]
        temp_data = []
        all_combinations_successful = True

        for K in strikes:
            for tau in maturities:
                maturity = int(tau * 365)
                todaysDate = ql.Date(17, 2, 2024)
                ql.Settings.instance().evaluationDate = todaysDate

                settlementDate = todaysDate
                day_count = ql.Actual365Fixed()
                rfr = ql.YieldTermStructureHandle(ql.FlatForward(settlementDate, r, day_count))
                div_yield = ql.YieldTermStructureHandle(ql.FlatForward(settlementDate, q, day_count))

                exercise = ql.EuropeanExercise(todaysDate + maturity)
                payoff = ql.PlainVanillaPayoff(ql.Option.Call, K)
                european_option = ql.VanillaOption(payoff, exercise)

                spot = ql.QuoteHandle(ql.SimpleQuote(S0))
                heston_process = ql.HestonProcess(rfr, div_yield, spot, V0, lamda, vbar, eta, rho)
                engine = ql.AnalyticHestonEngine(ql.HestonModel(heston_process), 1e-15, int(1e6))
                european_option.setPricingEngine(engine)

                try:
                    price = european_option.NPV()
                    if price > 0 and price + K >= S0:
                        iv = implied_volatility(price, S0, K, tau, r, 'c')
                        temp_data.append([eta, rho, lamda, vbar, V0, tau, K, price, iv])
                    else:
                        all_combinations_successful = False
                        break
                except:
                    all_combinations_successful = False
                    break
            
            if not all_combinations_successful:
                break

        if all_combinations_successful:
            option_data.extend(temp_data)
    heading_list = ['eta', 'rho', 'lambda', 'vbar', 'V0', 'tau', 'K', 'price', 'iv']
    df = pd.DataFrame(option_data, columns=heading_list)
    df.to_csv(filename + str(ID) + '.csv')
    return(ID)

In [ ]:
generateData(1, 10, 'TestHeston')

In [ ]:
# Below is the provided multithreaded code snippet for context:
filename = 'heston_option'
no_threads = os.cpu_count() - 4  # Adjusting the number of threads
pkgsize = 1000  # Number of samples per thread
dataSize = 100  # Total number of iterations/data chunks

In [ ]:
pool = mp.Pool(processes=no_threads)
pbar = tqdm(total=no_threads)
results = [pool.apply_async(generateData, args=(i, pkgsize, filename), callback=lambda _: pbar.update(1)) for i in range(dataSize)]
output = [p.get() for p in results]
pool.close()
pool.join()